# Part 5: The Functional API

`Sequential` can only stack layers in one line. The **Functional API** lets you wire layers into any shape: several inputs, branches that join, and shortcuts. You write each layer as a function and call it on a tensor, and Keras records the connections.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow start-up messages

import keras
from keras import layers

import numpy as np

from churn_data import make_splits

splits = make_splits(42)
x_train, y_train = splits.train.x.numpy(), splits.train.y.numpy()
x_val, y_val = splits.val.x.numpy(), splits.val.y.numpy()
print(x_train.shape, y_train.mean().round(3))

## Two inputs

The prepared table has 9 columns: 3 numbers first, then 6 one-hot columns for the categories. Give each group its own branch, then join them.

In [ ]:
keras.utils.set_random_seed(42)

numbers = keras.Input(shape=(3,), name="numbers")
categories = keras.Input(shape=(6,), name="categories")

a = layers.Dense(8, activation="relu", name="num_branch")(numbers)
b = layers.Dense(4, activation="relu", name="cat_branch")(categories)
joined = layers.Concatenate(name="join")([a, b])
hidden = layers.Dense(16, activation="relu", name="hidden")(joined)
score = layers.Dense(1, name="score")(hidden)

model = keras.Model(inputs=[numbers, categories], outputs=score, name="two_branch")
model.summary()

The summary has a column, **Connected to**, that shows the graph. Train it by passing one array per input, in a list or in a dictionary keyed by the input names.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(3e-3),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
)


def split(a):
    return [a[:, :3], a[:, 3:]]


history = model.fit(
    split(x_train), y_train,
    validation_data=(split(x_val), y_val),
    epochs=30, batch_size=64, verbose=0,
)
print("validation loss:", round(history.history["val_loss"][-1], 3), " parameters:", model.count_params())

# the dictionary form uses the names you gave the inputs
model.fit(
    {"numbers": x_train[:, :3], "categories": x_train[:, 3:]},
    y_train, epochs=1, batch_size=64, verbose=0,
)

## A shortcut connection

Here the output of the first hidden layer is added to the output of the second, so the signal has a direct path past one layer. This is the idea behind residual networks.

In [ ]:
keras.utils.set_random_seed(42)

inputs = keras.Input(shape=(9,))
h1 = layers.Dense(16, activation="relu")(inputs)
h2 = layers.Dense(16, activation="relu")(h1)
merged = layers.Add()([h1, h2])
outputs = layers.Dense(1)(merged)

skip = keras.Model(inputs, outputs, name="skip")
skip.compile(
    optimizer=keras.optimizers.Adam(3e-3),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
)
history = skip.fit(x_train, y_train, validation_data=(x_val, y_val),
                   epochs=30, batch_size=64, verbose=0)
print("validation loss:", round(history.history["val_loss"][-1], 3), " parameters:", skip.count_params())
print([layer.name for layer in skip.layers])

## Reading from the middle of a model

Because the graph is explicit, you can build a second model that ends at any tensor you named. This is how you pull out the features a network has learned.

In [ ]:
features = keras.Model(skip.input, h2)
print(features.predict(x_val[:3], verbose=0).shape)

## When to use which

| You need | Use |
|---|---|
| One straight stack of layers | `Sequential` |
| Several inputs or outputs, branches, shortcuts, shared layers | Functional API |
| Custom behaviour that the graph cannot express | a subclass of `keras.Model`, which is the closest to PyTorch's `nn.Module` |

## Your turn

Add a second output with a different head, for example a plain `Dense(1, activation="sigmoid")` named `probability`, and look at the new summary.